# TFLN traveling-wave MZM: closed form versus sections

In a thin-film lithium niobate (TFLN) modulator, the Pockels effect changes the effective index linearly with voltage, and no junction loads the electrode. The electrode is therefore a linear transmission line, and the small-signal electro-optic (EO) response has a closed form that accounts for velocity mismatch, RF loss and source and load reflections. Expressions of this type appear, for example, in H. Chung, W. S. C. Chang and E. L. Adler, *IEEE J. Quantum Electron.* **27**, 608 (1991), and in G. Ghione, *Semiconductor Devices for High-Speed Optoelectronics* (Cambridge University Press, 2009).

This example:

1. evaluates the closed-form small-signal response;
2. uses it to construct a 56 Gbaud eye diagram;
3. examines its limits when the index change is not linear in voltage;
4. compares it with a circuit model divided into 250 µm sections, built from library components.

Light leaving the line at time $t$ passed position $z$ at $t-(L-z)n_{g,\mathrm{opt}}/c$. Its phase follows the delay-weighted mean voltage $V_\mathrm{eff}$, which for a uniform line is a linear function of the two end voltages:

$$V_\mathrm{eff}=H_1V_1+H_2V_2,\qquad H_1=\frac{F_\mathrm{co}-e^{-\gamma L}F_\mathrm{ctr}}{1-e^{-2\gamma L}},\qquad H_2=\frac{F_\mathrm{ctr}-e^{-\gamma L}F_\mathrm{co}}{1-e^{-2\gamma L}},$$

$$F_\mathrm{co}=\frac{e^{-kL}-e^{-\gamma L}}{(\gamma-k)L},\qquad F_\mathrm{ctr}=\frac{1-e^{-(\gamma+k)L}}{(\gamma+k)L},\qquad k=\frac{j2\pi f\,n_{g,\mathrm{opt}}}{c}.$$

$F_\mathrm{co}$ is the contribution of the RF wave travelling with the light, limited by the mismatch between the RF phase index and the optical group index. $F_\mathrm{ctr}$ is that of the reflected, counter-propagating wave. For a matched load it reduces to the familiar $|(1-e^{-uL})/(uL)|$ with $u=\alpha+j2\pi f(n_\mathrm{RF}-n_{g,\mathrm{opt}})/c$. The same expression applies to any section of the line, which is used below.

In [ ]:
# Copyright 2026 Circulax contributors

import time
from itertools import pairwise

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from circulax import compile_circuit
from circulax.components import characteristic_rlgc_transmission_line, phase_shifter_factory
from circulax.components.base_component import component
from circulax.components.electronic import WAVE_PWL, Resistor, WaveformVoltageSource
from circulax.components.photonic import DirectionalCoupler, OpticalSource

jax.config.update("jax_enable_x64", True)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True})
c0 = 299_792_458.0
length_m, n_opt, wavelength_nm, dn_dv = 5e-3, 2.25, 1550.0, 1.5e-5  # dn_dv: |dn_eff/dV| per arm
k0 = 2 * np.pi * length_m * 1e9 / wavelength_nm  # phase per unit index change, rad
baud, samples_per_ui, preamble_bits, target_er_db = 56e9, 32, 16, 4.0
ui = 1 / baud
quadrature_phase = -np.pi / 2


def prbs7_bits(preamble_bits):
    state, pattern = 0x7F, []
    for _ in range(127):
        pattern.append(state & 1)
        state = (state >> 1) | (((state ^ (state >> 1)) & 1) << 6)
    return np.r_[np.full(preamble_bits, pattern[0]), pattern]


def band_limited_drive(bits, vpp, bandwidth=0.75 * baud, points_per_ui=16):
    """NRZ through a Gaussian filter (-3 dB at `bandwidth`), as PWL samples every UI/points_per_ui."""
    fine = 16 * points_per_ui
    nrz = vpp * (np.repeat(bits, fine) - 0.5)
    padded = np.r_[np.full(len(nrz), nrz[0]), nrz]  # settled preamble before t = 0
    f = np.fft.rfftfreq(len(padded), ui / fine)
    sigma = bandwidth / np.sqrt(2 * np.log(2))
    smooth = np.fft.irfft(np.fft.rfft(padded) * np.exp(-0.5 * (f / sigma) ** 2), len(padded))[len(nrz) :]
    t = np.arange(len(nrz)) * ui / fine
    return t[::16], smooth[::16]


def fold_eye(times, signal, *, flight_time=0.0):
    """Fold two UI with one central eye; skip the preamble and settling."""
    n_bits = (len(times) - 1) // samples_per_ui
    eye_ui = np.arange(2 * samples_per_ui + 1) / samples_per_ui
    fold_bits = np.arange(preamble_bits + 8, n_bits - 5)
    query = flight_time + (fold_bits[:, None] + 0.5 + eye_ui[None, :]) * ui
    return eye_ui, np.interp(query.ravel(), times, signal).reshape(query.shape)


bits = prbs7_bits(preamble_bits)
times = np.arange(len(bits) * samples_per_ui + 1) * ui / samples_per_ui

## Electrode

The illustrative 5 mm coplanar electrode has a 45 Ω characteristic impedance and a skin-effect loss of about 0.5 dB/cm at 1 GHz, growing as $\sqrt f$. The skin effect also adds an internal inductance, which keeps the RLGC data causal. Two designs are compared: a **matched** design (RF phase index 2.2 against an optical group index of 2.25, 50 Ω load), and a **walk-off + reflection** design (RF phase index 2.6, 30 Ω load) that shows where the closed form degrades. Both are driven from 50 Ω.

In [ ]:
r_dc, z_line = 300.0, 45.0  # ohm/m, ohm
r_skin = 2 * z_line * (0.5e2 * np.log(10) / 20) / np.sqrt(1e9)  # ~0.5 dB/cm at 1 GHz, growing as sqrt(f)


def electrode_rlgc(n_rf):
    """Per-metre R, L, G, C versus Hz; the skin-effect internal inductance keeps the data causal."""

    def r(f):
        return r_dc + r_skin * np.sqrt(f)

    def inductance(f):
        return z_line * n_rf / c0 + np.divide(r_skin * np.sqrt(f), 2 * np.pi * f, out=np.zeros_like(f), where=f > 0)

    def c(f):
        return np.full_like(f, n_rf / (z_line * c0))

    return {"R": r, "L": inductance, "G": 0.0, "C": c}


designs = {
    "matched": {"n_rf": 2.2, "load_r": 50.0},
    "walk-off + reflection": {"n_rf": 2.6, "load_r": 30.0},
}
source_r = 50.0

## Small-signal EO response

`section_veff` evaluates the closed form for any division of the line into sections, including the 50 Ω source and the load; the whole line is a single section. `time_response` passes a drive waveform through such a response by FFT.

In [ ]:
def propagation(f, n_rf):
    """Propagation constant gamma and characteristic impedance Zc of the electrode."""
    rlgc = electrode_rlgc(n_rf)
    f = np.maximum(np.asarray(f, dtype=float), 1e3)  # the DC limit, evaluated just above DC
    s = 2j * np.pi * f
    z, y = rlgc["R"](f) + s * rlgc["L"](f), s * rlgc["C"](f)
    return np.sqrt(z * y), np.sqrt(z / y)


def section_veff(f, edges, *, n_rf, load_r):
    """Closed-form effective voltage of each section, per volt of source, seen by light leaving the line.

    For a section of length l between z_a and z_b, V_eff = H1 V(z_a) + H2 V(z_b), with the closed-form
    H1, H2 of a uniform line. The light then still needs (L - z_b) n_opt / c to leave the line.
    """
    f = np.asarray(f, dtype=float)
    gamma, zc = propagation(f, n_rf)
    k = 2j * np.pi * f * n_opt / c0
    ch, sh = np.cosh(gamma * length_m), np.sinh(gamma * length_m)
    v_load = 1 / (ch + zc * sh / load_r + source_r * (sh / zc + ch / load_r))

    def v(z):  # voltage at z, from the load end
        return v_load * (np.cosh(gamma * (length_m - z)) + zc / load_r * np.sinh(gamma * (length_m - z)))

    out = []
    for za, zb in pairwise(edges):
        l = zb - za
        e = np.exp(-gamma * l)
        co = np.exp(-k * l) * -np.expm1(-(gamma - k) * l) / ((gamma - k) * l)
        counter = -np.expm1(-(gamma + k) * l) / ((gamma + k) * l)
        h1, h2 = (co - e * counter) / (1 - e**2), (counter - e * co) / (1 - e**2)
        out.append((h1 * v(za) + h2 * v(zb)) * np.exp(-k * (length_m - zb)))
    return np.array(out)


def time_response(transfer, drive_t, drive_v):
    """Response at `times` of the PWL drive through `transfer(f)`; the record is padded so it is periodic and settled."""
    record = len(bits) * ui
    dt = ui / 64
    t = np.arange(0.0, 4 * record, dt)
    v = np.interp(t, np.r_[drive_t, record, 3 * record, 3 * record + ui], np.r_[drive_v, drive_v[-1], drive_v[-1], drive_v[0]])
    f = np.fft.rfftfreq(len(t), dt)
    h = np.atleast_2d(transfer(f))
    out = np.fft.irfft(np.fft.rfft(v) * h, len(t), axis=-1)
    return np.array([np.interp(times, t, row) for row in out])


f_plot = np.linspace(0.0, 120e9, 601)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, design in designs.items():
    eo = section_veff(f_plot, [0.0, length_m], **design)[0]
    axes[0].plot(f_plot[1:] / 1e9, c0 * propagation(f_plot[1:], design["n_rf"])[0].imag / (2 * np.pi * f_plot[1:]), label=name)
    axes[1].plot(f_plot / 1e9, 20 * np.log10(np.abs(eo / eo[0])), label=name)
    report = ", ".join(
        f"{fr / 1e9:.0f} GHz {20 * np.log10(abs(section_veff([fr], [0, length_m], **design)[0, 0] / eo[0])):.2f} dB"
        for fr in (28e9, 56e9, 112e9)
    )
    print(f"{name}: EO response {report}")
axes[0].axhline(n_opt, color="k", linestyle="--", label="optical group index")
axes[0].set(xlabel="Frequency (GHz)", ylabel="RF phase index", ylim=(2.1, 2.8))
axes[1].set(xlabel="Frequency (GHz)", ylabel="EO response (dB)", ylim=(-6, 1))
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

## Eye diagram from the closed form

For a linear index change, each arm's phase is $k_0\,(dn/dV)\,V_\mathrm{eff}(t)$ with $k_0=2\pi L/\lambda$. Push-pull doubles the differential phase, and the MZM cosine transfer is applied to the total phase afterwards, so the optical nonlinearity of the MZM itself is retained. The source swing is set for a 4 dB static extinction ratio (ER). A Gaussian-filtered PRBS7 drive with a −3 dB bandwidth of 0.75 times the baud rate represents a band-limited driver and is used for every model below.

In [ ]:
design = designs["matched"]
eo_dc = section_veff([0.0], [0.0, length_m], **design)[0, 0].real  # effective volts per source volt at DC


def mzm_power(delta_theta):
    """Normalized MZM output for a push-pull differential phase, biased at quadrature."""
    return (1 - np.cos(quadrature_phase + delta_theta)) / 2


swing = np.linspace(0.1, 8.0, 400)
er = 10 * np.log10(mzm_power(-k0 * dn_dv * eo_dc * swing) / mzm_power(k0 * dn_dv * eo_dc * swing))
vpp = float(np.interp(target_er_db, np.abs(er), swing))
drive_t, drive_v = band_limited_drive(bits, vpp)
veff = time_response(lambda f: section_veff(f, [0.0, length_m], **design)[0], drive_t, drive_v)[0]
closed_power = mzm_power(2 * k0 * dn_dv * veff)
print(f"Source swing for {target_er_db:.0f} dB ER: {vpp:.2f} Vpp")
eye_ui, eye = fold_eye(times, closed_power)
plt.figure()
plt.plot(eye_ui, eye.T, color="C0", alpha=0.15, lw=0.8)
plt.xlabel("Time (UI)")
plt.ylabel("Normalized power")
plt.title("56 Gbaud eye from the closed-form small-signal response")
plt.xlim(0, 2)
plt.tight_layout()
plt.show()

## Pitfall: nonlinear index change

The closed form relies on superposition: the phase is taken as the index change at the path-averaged voltage. This is exact only when $\Delta n_\mathrm{eff}$ is linear in $V$. Otherwise the path average of $\Delta n_\mathrm{eff}(V(z,t))$ differs from $\Delta n_\mathrm{eff}(V_\mathrm{eff})$, by approximately the curvature of $\Delta n_\mathrm{eff}$ times the variance of the voltage seen along the optical path.

An illustrative saturating index change, $\Delta n_\mathrm{eff}=(dn/dV)\,V_\mathrm{sat}\tanh(V/V_\mathrm{sat})$, is used. It is odd in $V$ because push-pull cancels even-order terms; it is not a model of lithium niobate, whose Pockels response is very linear. Because the electrode itself remains linear, an exact reference is available: the same closed form applied over 25 µm sections, with the index change evaluated per section. The convergence plot confirms that this reference is converged.

Three approximations are compared with it:

- the **linear closed form**, which ignores the nonlinearity;
- the **closed form with $\Delta n_\mathrm{eff}(V_\mathrm{eff})$** over the whole line;
- the **closed form per 250 µm section**, with $\Delta n_\mathrm{eff}$ evaluated per section.

In [ ]:
v_sat = 2.6  # V; illustrative saturation, about 3% compression at the 0.75 V peak line voltage


def delta_neff(v, dn_dv=dn_dv, v_sat=v_sat, sign=1.0):
    """Illustrative odd, saturating index change; push-pull cancels even-order terms."""
    return sign * dn_dv * v_sat * jnp.tanh(v / v_sat)


def arm_phase(veff_sections):
    """Phase of one arm from per-section effective voltages (equal-length sections)."""
    return k0 * np.mean(np.asarray(delta_neff(veff_sections)), axis=0)


def sections(length_um):
    return np.linspace(0.0, length_m, round(length_m * 1e6 / length_um) + 1)


section_lengths_um = (2500, 1000, 500, 250, 125)
reference_um = 25  # converged reference, see the convergence plot
results = {}
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, design in designs.items():
    by_section = {
        um: time_response(lambda f, um=um, design=design: section_veff(f, sections(um), **design), drive_t, drive_v)
        for um in (5000, *section_lengths_um, reference_um)
    }
    exact = arm_phase(by_section[reference_um])
    peak = np.max(np.abs(exact))

    def rms(phase, exact=exact, peak=peak):
        return float(np.sqrt(np.mean((phase - exact) ** 2)) / peak)

    results[name] = {
        "linear closed form": rms(k0 * dn_dv * by_section[5000][0]),
        "closed form, nonlinear dn(V_eff)": rms(arm_phase(by_section[5000])),
        "closed form per 250 um section": rms(arm_phase(by_section[250])),
    }
    ax.loglog(section_lengths_um, [rms(arm_phase(by_section[um])) for um in section_lengths_um], "o-", label=name)
    ax.axhline(results[name]["closed form, nonlinear dn(V_eff)"], color=ax.lines[-1].get_color(), linestyle=":")
ax.set(xlabel="Section length (um)", ylabel="RMS phase error / peak", title="Closed form per section (dotted: whole line)")
ax.minorticks_off()
ax.set_xticks(section_lengths_um, [str(um) for um in section_lengths_um])
ax.legend()
plt.tight_layout()
plt.show()
for name, errors in results.items():
    print(f"{name}: " + ", ".join(f"{label} {value:.1e}" for label, value in errors.items()))
phase_dc = k0 * np.asarray(delta_neff(eo_dc * vpp / 2))
print(
    f"Static ER at {vpp:.2f} Vpp: {abs(10 * np.log10(mzm_power(2 * phase_dc) / mzm_power(-2 * phase_dc))):.2f} dB with the saturating index change (calibrated linearly for {target_er_db:.1f} dB)"
)

Ignoring the nonlinearity is the largest error, about 2 % of the peak phase, and it shifts the extinction ratio. Applying the nonlinearity to the whole-line $V_\mathrm{eff}$ is accurate for the matched design but degrades by about twenty times with walk-off and reflections, and its error cannot be known without a reference. Dividing the line into 250 µm sections reduces it by about two orders of magnitude in both designs, and the error falls as the square of the section length.

No closed form exists when the electrode itself is nonlinear, for example when it is loaded by a voltage-dependent junction capacitance as in the [traveling-wave MZM tutorial](traveling_wave_mzm.md). The same applies to bias-dependent optical loss acting together with RF loading, and to non-uniform electrodes. A sectioned circuit model is then required.

## Sectioned circuit model

The circuit model uses only library components. Each 250 µm section is two 125 µm halves of `characteristic_rlgc_transmission_line`, fitted once and reused. In each arm, a `phase_shifter_factory(..., junction=None)` section with the saturating index change senses the centre tap and applies the optical transit delay. Arm B uses the opposite sign.

This model samples the voltage at the centre of each section rather than averaging it with the closed form, so its spatial error is second order in the section length. Unlike the closed form, however, it extends unchanged to loaded electrodes, nonlinear drivers and bias networks. A time step of UI/128 is used; at UI/64 the time-step error is comparable to the effects being compared. The result is compared with the exact reference in normalized output power.

In [ ]:
design = designs["matched"]
section_um, n_sections = 250.0, 20
half_section = characteristic_rlgc_transmission_line(
    np.r_[0.0, np.geomspace(1e5, 250e9, 400)], **electrode_rlgc(design["n_rf"]), length=section_um * 0.5e-6
)
PockelsSection = phase_shifter_factory(delta_neff=delta_neff, name="PockelsSection")


@component(ports=("p1", "p2"), states=("a1", "a2"), holomorphic=True)
def PhaseBias(signals, phase=0.0):  # noqa: N802
    """Matched, reciprocal phase-only optical element; field impedance is 1 ohm."""
    transmission = jnp.exp(1j * phase)
    b1, b2 = transmission * signals.a2, transmission * signals.a1
    return {
        "p1": signals.a1 - b1,
        "p2": signals.a2 - b2,
        "a1": signals.p1 - signals.a1 - b1,
        "a2": signals.p2 - signals.a2 - b2,
    }, {}


def sectioned_mzm(drive_t, drive_v):
    """Driver -> 20 electrode sections -> load; each arm has one phase shifter per section at its centre tap."""
    ins = {
        "GND": {"component": "ground"},
        "VS": {"component": "drive", "settings": {"kind": WAVE_PWL, "pwl_t": jnp.asarray(drive_t), "pwl_v": jnp.asarray(drive_v)}},
        "RS": {"component": "R", "settings": {"R": source_r}},
        "RL": {"component": "R", "settings": {"R": design["load_r"]}},
        "LASER": {"component": "laser", "settings": {"power": 1.0}},
        "SPLIT": {"component": "coupler", "settings": {"coupling": 0.5}},
        "COMBINE": {"component": "coupler", "settings": {"coupling": 0.5}},
        "BIAS": {"component": "bias", "settings": {"phase": quadrature_phase}},
    }
    con = {"VS,p1": "RS,p1", "VS,p2": "GND,p1", "RL,p2": "GND,p1", "LASER,p1": "SPLIT,p1", "LASER,p2": "GND,p1"}
    for dump in ("DUMP_IN", "DUMP_OUT", "LOAD"):
        ins[dump] = {"component": "R", "settings": {"R": 1.0}}
        con[f"{dump},p2"] = "GND,p1"
    con.update({"SPLIT,p2": "DUMP_IN,p1", "COMBINE,p4": "DUMP_OUT,p1", "COMBINE,p3": "LOAD,p1", "BIAS,p2": "COMBINE,p2"})
    node = "RS,p2"
    for k in range(n_sections):
        ins[f"E{k}a"], ins[f"E{k}b"] = {"component": "line"}, {"component": "line"}
        con[f"E{k}a,p1"], con[f"E{k}b,p1"] = node, f"E{k}a,p2"
        node = f"E{k}b,p2"
        for arm, sign, start, end in (("A", 1.0, "SPLIT,p3", "COMBINE,p1"), ("B", -1.0, "SPLIT,p4", "BIAS,p1")):
            settings = {"length_um": section_um, "n_group": n_opt, "neff": 1.9, "wavelength_nm": wavelength_nm, "sign": sign}
            ins[f"{arm}{k}"] = {"component": "pockels", "settings": settings}
            con[f"{arm}{k},anode"], con[f"{arm}{k},cathode"] = f"E{k}a,p2", "GND,p1"
            con[f"{arm}{k},o1"] = start if k == 0 else f"{arm}{k - 1},o2"
            if k == n_sections - 1:
                con[f"{arm}{k},o2"] = end
    con[f"{node}"] = "RL,p1"
    net = {"instances": ins, "connections": con, "ports": {"output": "LOAD,p1"}}
    models = {
        "drive": WaveformVoltageSource,
        "R": Resistor,
        "line": half_section,
        "pockels": PockelsSection,
        "laser": OpticalSource,
        "coupler": DirectionalCoupler,
        "bias": PhaseBias,
    }
    return compile_circuit(net, models, is_complex=True, rtol=1e-9, atol=1e-10)


mzm = sectioned_mzm(drive_t, drive_v)
started = time.perf_counter()
solution = mzm.transient(
    t0=0.0, t1=float(times[-1]), dt0=ui / 128, saveat=jnp.asarray(times), max_steps=200 * len(bits) + 100, throw=True
)
sectioned_power = np.abs(np.asarray(mzm.port(solution.ys, "output"))) ** 2
print(f"Sectioned transient: {time.perf_counter() - started:.1f} s")

veff_reference = time_response(lambda f: section_veff(f, sections(reference_um), **design), drive_t, drive_v)
exact_power = mzm_power(2 * arm_phase(veff_reference))
nonlinear_closed_power = mzm_power(2 * arm_phase(veff[None, :]))
assert np.all(np.isfinite(sectioned_power))
assert np.min(sectioned_power) >= 0
assert np.max(sectioned_power) <= 1.001
sectioned_rms_error = np.sqrt(np.mean((sectioned_power - exact_power) ** 2))
assert sectioned_rms_error < 1e-3, f"Sectioned circuit RMS power error: {sectioned_rms_error:.3g}"
for label, power in (
    ("linear closed form", closed_power),
    ("closed form, dn(V_eff)", nonlinear_closed_power),
    ("sectioned circuit", sectioned_power),
):
    print(f"{label}: RMS normalized-power error vs exact {np.sqrt(np.mean((power - exact_power) ** 2)):.1e}")
eye_ui, eye_sectioned = fold_eye(times, sectioned_power, flight_time=length_m * n_opt / c0)
plt.figure()
plt.plot(eye_ui, eye_sectioned.T, color="C2", alpha=0.15, lw=0.8)
plt.xlabel("Time (UI)")
plt.ylabel("Normalized power")
plt.title("56 Gbaud eye, 20 x 250 um sections with saturating index change")
plt.xlim(0, 2)
plt.tight_layout()
plt.show()

The sectioned circuit agrees with the exact result much more closely than the linear closed form. For this matched, unloaded design, applying the nonlinearity to the whole-line $V_\mathrm{eff}$ remains the most accurate approximation. The value of the sectioned model lies in its generality and in providing a reference where no closed form applies.